In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tsa.stattools import adfuller
from dwtest import dwtest
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro
from itertools import combinations, product
import math
from pandas_datareader import data as pdr
from tqdm import tqdm
from TDL_utilities.utilities_DL import *
import os

In [2]:
def write_file(folder_name='', file_name='', sheet_name='', data=None):
      if folder_name=='':
            if os.path.exists(file_name):
                  with pd.ExcelWriter(file_name, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
                              data.to_excel(writer, sheet_name = sheet_name)
            else: 
                  with pd.ExcelWriter(file_name,engine = 'openpyxl', mode = 'w') as writer:
                              data.to_excel(writer, sheet_name = sheet_name)
      elif not os.path.exists(folder_name):
            os.makedirs(folder_name)
            if os.path.exists(folder_name+'/'+file_name):
                        with pd.ExcelWriter(folder_name+'/'+file_name, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
                              data.to_excel(writer, sheet_name = sheet_name)
            else: 
                        with pd.ExcelWriter(folder_name+'/'+file_name, engine = 'openpyxl', mode = 'w') as writer:
                             data.to_excel(writer, sheet_name = sheet_name)
      else:
            if os.path.exists(folder_name+'/'+file_name):
                        with pd.ExcelWriter(folder_name+'/'+file_name, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
                              data.to_excel(writer, sheet_name = sheet_name)
            else: 
                        with pd.ExcelWriter(folder_name+'/'+file_name, engine = 'openpyxl', mode = 'w') as writer:
                              data.to_excel(writer, sheet_name = sheet_name)


In [3]:
# connect to SQL
from sqlalchemy import create_engine
from sqlalchemy.types import String, NVARCHAR, Text, DateTime
from urllib.parse import quote_plus
# Connect to the SQL server, Build the connection string
server = 'LAPTOP-JS537DIH'
database = 'Lending_Club'
conn_string = 'mssql+pyodbc://' + '@' + server + '/' + database + '?driver=ODBC+Driver+17+for+SQL+Server&charset=UTF8'
engine = create_engine(conn_string, fast_executemany=True)

In [4]:
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='darkgrid')

In [5]:
sql_PD = '''
	with data_full as(
		select year(last_pymnt_d) [year], DATEPART(QUARTER, last_pymnt_d) quarter, count(*) num
		from Lending_Club..LendingClub_PD
		where last_pymnt_d is not null
		group by year(last_pymnt_d), DATEPART(QUARTER, last_pymnt_d)
	),
	data_pd as (
		select year(last_pymnt_d) [year], DATEPART(QUARTER, last_pymnt_d) quarter, count(*) num_default
		from Lending_Club..LendingClub_PD
		where loan_status in ('Default', 'Charged Off', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)') and  last_pymnt_d is not null
		group by year(last_pymnt_d), DATEPART(QUARTER, last_pymnt_d)
	)
	select a.year, a.quarter, a.num, b.num_default
	from data_full a left join data_pd b on a.year = b.year and a.quarter = b.quarter
	order by a.year, a.quarter 


'''
df_PD = pd.read_sql(sql=sql_PD, con = engine)
df_PD.head()

,year,quarter,num,num_default
0,2007,4,2,1
1,2008,1,37,19
2,2008,2,73,34
3,2008,3,108,58
4,2008,4,117,74


In [6]:
df_PD['PD'] = df_PD['num_default']/df_PD['num']

In [7]:
def logit_PD(PD):
    if PD == 0:
        return np.log(0.0001 / (1 - 0.0001))
    return np.log(PD / (1 - PD))

def PD(logit_PD):
    return np.exp(logit_PD) / (np.exp(logit_PD) + 1)

def VIF_test(X):
    VIFs = []
    for col in X.columns:
        model = sm.OLS(X[col], sm.add_constant(X.drop(col, axis = 1))).fit()
        R2 = pd.read_html(model.summary().tables[0].as_html())[0].iloc[0, 3]
        VIF = 1 / (1 - R2)
        VIFs.append(VIF)
    return np.max(VIFs)

def float_list(original_list):
    return [float(i) for i in original_list]

In [8]:
df_PD = df_PD[(df_PD['year']>=2010) & (df_PD['year']<2020)]
df_PD.reset_index(drop=True, inplace=True)
df_PD['ln_PD'] = df_PD['PD'].apply(logit_PD)
df_PD

,year,quarter,num,num_default,PD,ln_PD
0,2010,1,432,136,0.3148,-0.7777
1,2010,2,446,144,0.3229,-0.7406
2,2010,3,660,177,0.2682,-1.0039
3,2010,4,882,223,0.2528,-1.0836
4,2011,1,1363,186,0.1365,-1.8450
5,2011,2,1407,271,0.1926,-1.4331
6,2011,3,1424,354,0.2486,-1.1061
7,2011,4,1554,378,0.2432,-1.1350
8,2012,1,2329,451,0.1936,-1.4265
9,2012,2,2599,526,0.2024,-1.3715


In [9]:
target = 'ln_PD'
num_var = 2
len_OOT = 4
from datetime import datetime
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")

In [10]:
series = ['GDPC1', 'UNRATE', 'CPIAUCSL', 'FEDFUNDS', 'DSPIC96', 'USSTHPI', 'CDSP', 'TDSP', 'DRCLACBS', 'CORCACBS', 'PSAVERT', 'TERMCBCCALLNS','IR', 'IQ', 'IC131', 'IS231', 'POPTHM', 
           'TTLHHM156N' ,'CSUSHPINSA', 'DCOILBRENTEU', 'GASREGW', 'DCOILWTICO', 'ECIWAG', 'NASDAQCOM', 'A782RC1Q027SBEA', 'GCE', 'GGSAVE', 'GFDEGDQ188S', 'FGRECPT', 'FGEXPND', 
           'TB3MS', 'DGS2']
df_FED = pdr.DataReader(series, 'fred', start='1989-01-01', end='2020-12-31')

In [11]:
df_FED = df_FED.resample('QE').last()

In [12]:
df_FED.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 128 entries, 1989-03-31 to 2020-12-31
Freq: QE-DEC
Data columns (total 32 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   GDPC1            128 non-null    float64
 1   UNRATE           128 non-null    float64
 2   CPIAUCSL         128 non-null    float64
 3   FEDFUNDS         128 non-null    float64
 4   DSPIC96          128 non-null    float64
 5   USSTHPI          128 non-null    float64
 6   CDSP             64 non-null     float64
 7   TDSP             64 non-null     float64
 8   DRCLACBS         128 non-null    float64
 9   CORCACBS         128 non-null    float64
 10  PSAVERT          128 non-null    float64
 11  TERMCBCCALLNS    105 non-null    float64
 12  IR               128 non-null    float64
 13  IQ               128 non-null    float64
 14  IC131            122 non-null    float64
 15  IS231            114 non-null    float64
 16  POPTHM           128 non-null 

In [13]:
print(df_FED.head().to_markdown())

| DATE                |    GDPC1 |   UNRATE |   CPIAUCSL |   FEDFUNDS |   DSPIC96 |   USSTHPI |   CDSP |   TDSP |   DRCLACBS |   CORCACBS |   PSAVERT |   TERMCBCCALLNS |   IR |   IQ |   IC131 |   IS231 |   POPTHM |   TTLHHM156N |   CSUSHPINSA |   DCOILBRENTEU |   GASREGW |   DCOILWTICO |   ECIWAG |   NASDAQCOM |   A782RC1Q027SBEA |     GCE |   GGSAVE |   GFDEGDQ188S |   FGRECPT |   FGEXPND |   TB3MS |   DGS2 |
|:--------------------|---------:|---------:|-----------:|-----------:|----------:|----------:|-------:|-------:|-----------:|-----------:|----------:|----------------:|-----:|-----:|--------:|--------:|---------:|-------------:|-------------:|---------------:|----------:|-------------:|---------:|------------:|------------------:|--------:|---------:|--------------:|----------:|----------:|--------:|-------:|
| 1989-03-31 00:00:00 |  9771.73 |      5   |      122.2 |       9.85 |    7087.1 |    157.29 |    nan |    nan |       3.43 |       1.57 |       9.4 |             nan | 91

In [14]:
percent_change_col = ['GDPC1', 'CPIAUCSL', 'DSPIC96', 'USSTHPI', 'IR', 'IQ', 'IC131', 'IS231', 'POPTHM', 'TTLHHM156N', 'CSUSHPINSA', 'DCOILBRENTEU', 'GASREGW', 'DCOILWTICO', 
                      'ECIWAG', 'NASDAQCOM', 'A782RC1Q027SBEA', 'GCE', 'GGSAVE', 'GFDEGDQ188S', 'FGRECPT']

In [15]:
df_FED_pct = pd.DataFrame()
for i in df_FED.columns:
    if i not in percent_change_col:
        df_FED_pct[i] = df_FED[i]

for i in percent_change_col:
    df_FED_pct[i] = df_FED[i].pct_change(periods=4) * 100

In [16]:
print(df_FED_pct.describe().to_string())

        UNRATE  FEDFUNDS    CDSP    TDSP  DRCLACBS  CORCACBS  PSAVERT  TERMCBCCALLNS    FGEXPND    TB3MS     DGS2    GDPC1  CPIAUCSL  DSPIC96  USSTHPI       IR       IQ    IC131    IS231   POPTHM  TTLHHM156N  CSUSHPINSA  DCOILBRENTEU  GASREGW  DCOILWTICO  ECIWAG  NASDAQCOM  A782RC1Q027SBEA      GCE      GGSAVE  GFDEGDQ188S  FGRECPT
count 128.0000  128.0000 64.0000 64.0000  128.0000  128.0000 128.0000       105.0000   128.0000 128.0000 128.0000 124.0000  124.0000 124.0000 124.0000 124.0000 124.0000 118.0000 110.0000 124.0000    124.0000    124.0000      124.0000 118.0000    124.0000 76.0000   124.0000         124.0000 124.0000    124.0000     124.0000 124.0000
mean    5.8695    3.0082  6.0354 13.0119    3.1615    2.5344   6.2008        13.6941 2,822.4216   2.7745   3.3748   2.3673    2.4068   2.7647   3.5148   1.1391   0.8978   2.8560   1.4381   0.9501      0.9670      3.7071        8.4156   3.8069      7.0513  2.4796    13.8446           3.6969   4.1248    711.0755       3.0612   4.406

In [ ]:
write_file(folder_name = target[3:]+'_'+current_time+'_result/',file_name='Result_'+target[3:]+'_'+ str(num_var) +'_var'+'_'+current_time+'.xlsx',sheet_name='data_FED_pct',data=df_FED_pct)

In [44]:
for i in df_FED_pct.columns:
    df_FED_pct[i + "_lag1"] = df_FED_pct[i].shift(1)
    df_FED_pct[i + "_lag2"] = df_FED_pct[i].shift(2)

In [45]:
df_FED_pct = df_FED_pct[df_FED_pct.index >= pd.to_datetime('2010-01-01')]

In [67]:
macro_info = [

    # =========================================================
    # 1. ECONOMIC GROWTH
    # =========================================================
    [
        'GDPC1',
        'Economic Growth',
        'Real Gross Domestic Product',
        'GDP thực của Mỹ, phản ánh quy mô và tăng trưởng của nền kinh tế.',
        '-',
        'Tăng trưởng kinh tế tốt hơn -> thu nhập và việc làm cải thiện -> PD giảm.'
    ],

    # =========================================================
    # 2. LABOR MARKET
    # =========================================================
    [
        'UNRATE',
        'Labor Market',
        'Unemployment Rate',
        'Tỷ lệ thất nghiệp của Mỹ, phản ánh tình trạng thị trường lao động.',
        '+',
        'Thất nghiệp tăng -> thu nhập và khả năng trả nợ suy giảm -> PD tăng.'
    ],

    # =========================================================
    # 3. INFLATION
    # =========================================================
    [
        'CPIAUCSL',
        'Inflation',
        'Consumer Price Index for All Urban Consumers: All Items',
        'Chỉ số giá tiêu dùng CPI, phản ánh mức giá chung của nền kinh tế Mỹ.',
        '+',
        'Lạm phát tăng -> chi phí sinh hoạt tăng và thu nhập thực giảm -> PD có xu hướng tăng.'
    ],

    # =========================================================
    # 4. MONETARY POLICY
    # =========================================================
    [
        'FEDFUNDS',
        'Monetary Policy',
        'Federal Funds Effective Rate',
        'Lãi suất Federal Funds hiệu dụng, phản ánh điều kiện và chính sách tiền tệ của Mỹ.',
        '+',
        'Lãi suất tăng -> điều kiện tín dụng thắt chặt và chi phí vay tăng -> PD tăng.'
    ],

    # =========================================================
    # 5. HOUSEHOLD INCOME
    # =========================================================
    [
        'DSPIC96',
        'Household Income',
        'Real Disposable Personal Income',
        'Thu nhập khả dụng thực của cá nhân sau thuế và đã điều chỉnh lạm phát.',
        '-',
        'Thu nhập khả dụng thực tăng -> khả năng thanh toán nợ cải thiện -> PD giảm.'
    ],

    # =========================================================
    # 6. HOUSEHOLD SAVING
    # =========================================================
    [
        'PSAVERT',
        'Household Saving',
        'Personal Saving Rate',
        'Tỷ lệ tiết kiệm cá nhân trên thu nhập khả dụng.',
        '-',
        'Tỷ lệ tiết kiệm tăng -> bộ đệm tài chính của hộ gia đình tăng -> PD giảm.'
    ],

    # =========================================================
    # 7. HOUSEHOLD DEBT BURDEN
    # =========================================================
    [
        'CDSP',
        'Household Debt Burden',
        'Consumer Debt Service Payments as a Percent of Disposable Personal Income',
        'Tỷ lệ nghĩa vụ trả nợ tiêu dùng trên thu nhập khả dụng của hộ gia đình.',
        '+',
        'Gánh nặng trả nợ tiêu dùng tăng -> áp lực dòng tiền tăng -> PD tăng.'
    ],

    [
        'TDSP',
        'Household Debt Burden',
        'Household Debt Service Payments as a Percent of Disposable Personal Income',
        'Tỷ lệ tổng nghĩa vụ trả nợ hộ gia đình trên thu nhập khả dụng.',
        '+',
        'Tổng gánh nặng trả nợ hộ gia đình tăng -> khả năng trả nợ suy giảm -> PD tăng.'
    ],

    # =========================================================
    # 8. CREDIT RISK
    # =========================================================
    [
        'DRCLACBS',
        'Credit Risk',
        'Delinquency Rate on Consumer Loans, All Commercial Banks',
        'Tỷ lệ các khoản vay tiêu dùng bị quá hạn tại các ngân hàng thương mại Mỹ.',
        '+',
        'Delinquency của thị trường tăng -> chất lượng tín dụng suy giảm -> PD LendingClub tăng.'
    ],

    [
        'CORCACBS',
        'Credit Risk',
        'Charge-Off Rate on Consumer Loans, All Commercial Banks',
        'Tỷ lệ các khoản vay tiêu dùng bị charge-off tại các ngân hàng thương mại Mỹ.',
        '+',
        'Charge-off rate tăng -> môi trường rủi ro tín dụng xấu đi -> PD tăng.'
    ],

    # =========================================================
    # 9. CONSUMER CREDIT COST
    # =========================================================
    [
        'TERMCBCCALLNS',
        'Consumer Credit Cost',
        'Commercial Bank Interest Rate on Credit Card Plans, All Accounts',
        'Lãi suất áp dụng cho các tài khoản thẻ tín dụng tại ngân hàng thương mại Mỹ.',
        '+',
        'Lãi suất tín dụng tiêu dùng tăng -> chi phí phục vụ nợ tăng -> PD tăng.'
    ],

    # =========================================================
    # 10. HOUSING MARKET
    # =========================================================
    [
        'USSTHPI',
        'Housing Market',
        'All-Transactions House Price Index for the United States',
        'Chỉ số giá nhà trên toàn nước Mỹ do FHFA công bố.',
        '-',
        'Giá nhà tăng thường phản ánh tài sản hộ gia đình và điều kiện kinh tế tốt hơn -> PD giảm.'
    ],

    [
        'CSUSHPINSA',
        'Housing Market',
        'S&P Cotality Case-Shiller U.S. National Home Price Index',
        'Chỉ số giá nhà Case-Shiller trên toàn nước Mỹ.',
        '-',
        'Giá nhà tăng -> household wealth và giá trị tài sản tăng -> PD có xu hướng giảm.'
    ],

    # =========================================================
    # 11. TRADE PRICES
    # =========================================================
    [
        'IR',
        'Trade Prices',
        'Import Price Index (End Use): All Commodities',
        'Chỉ số giá hàng hóa nhập khẩu của Mỹ. Đây không phải Interest Rate.',
        '+',
        'Giá nhập khẩu tăng có thể tạo áp lực lạm phát và chi phí sinh hoạt -> PD tăng.'
    ],

    [
        'IQ',
        'Trade Prices',
        'Export Price Index (End Use): All Commodities',
        'Chỉ số giá hàng hóa xuất khẩu của Mỹ.',
        '+/-',
        'Tác động lên PD không rõ ràng vì phụ thuộc vào tăng trưởng xuất khẩu, thu nhập và lạm phát.'
    ],

    [
        'IC131',
        'Trade / Transportation Prices',
        'Inbound Price Index: Air Freight',
        'Chỉ số giá dịch vụ vận chuyển hàng hóa bằng đường hàng không vào Mỹ.',
        '+/-',
        'Quan hệ với consumer PD mang tính gián tiếp và không có dấu kỳ vọng rõ ràng.'
    ],

    [
        'IS231',
        'Trade / Transportation Prices',
        'Outbound Price Index: Air Freight',
        'Chỉ số giá dịch vụ vận chuyển hàng hóa bằng đường hàng không từ Mỹ ra nước ngoài.',
        '+/-',
        'Quan hệ với consumer PD mang tính gián tiếp và không có dấu kỳ vọng rõ ràng.'
    ],

    # =========================================================
    # 12. DEMOGRAPHICS
    # =========================================================
    [
        'POPTHM',
        'Demographics',
        'Population',
        'Ước tính dân số Mỹ.',
        '+/-',
        'Quy mô dân số không có quan hệ trực tiếp rõ ràng với xác suất vỡ nợ.'
    ],

    [
        'TTLHHM156N',
        'Demographics',
        'Household Estimates',
        'Ước tính tổng số hộ gia đình tại Mỹ.',
        '+/-',
        'Số hộ gia đình không có dấu tác động trực tiếp rõ ràng lên PD.'
    ],

    # =========================================================
    # 13. ENERGY PRICES
    # =========================================================
    [
        'DCOILBRENTEU',
        'Energy Prices',
        'Crude Oil Prices: Brent - Europe',
        'Giá dầu thô Brent giao ngay, benchmark quan trọng của thị trường dầu thế giới.',
        '+',
        'Giá dầu tăng -> chi phí năng lượng và sinh hoạt tăng -> PD có xu hướng tăng.'
    ],

    [
        'DCOILWTICO',
        'Energy Prices',
        'Crude Oil Prices: West Texas Intermediate (WTI)',
        'Giá dầu thô WTI giao ngay tại Cushing, Oklahoma.',
        '+',
        'Giá dầu tăng -> chi phí năng lượng và áp lực lạm phát tăng -> PD có xu hướng tăng.'
    ],

    [
        'GASREGW',
        'Energy Prices',
        'U.S. Regular All Formulations Retail Gasoline Prices',
        'Giá bán lẻ xăng regular bình quân tại Mỹ.',
        '+',
        'Giá xăng tăng -> chi phí sinh hoạt tăng và disposable cash flow giảm -> PD tăng.'
    ],

    # =========================================================
    # 14. LABOR COST / WAGES
    # =========================================================
    [
        'ECIWAG',
        'Labor Cost / Wages',
        'Employment Cost Index: Wages and Salaries: Private Industry Workers',
        'Chỉ số chi phí tiền lương của lao động khu vực tư nhân Mỹ.',
        '-',
        'Tiền lương tăng -> thu nhập và khả năng trả nợ của hộ gia đình cải thiện -> PD giảm.'
    ],

    # =========================================================
    # 15. FINANCIAL MARKET
    # =========================================================
    [
        'NASDAQCOM',
        'Financial Market',
        'NASDAQ Composite Index',
        'Chỉ số NASDAQ Composite, phản ánh diễn biến thị trường cổ phiếu NASDAQ.',
        '-',
        'Thị trường chứng khoán cải thiện thường đi cùng điều kiện kinh tế và tài chính tốt hơn -> PD giảm.'
    ],

    # =========================================================
    # 16. GOVERNMENT INVESTMENT
    # =========================================================
    [
        'A782RC1Q027SBEA',
        'Government Investment',
        'Gross Government Investment',
        'Tổng đầu tư của khu vực chính phủ Mỹ, bao gồm chính phủ liên bang, bang và địa phương.',
        '+/-',
        'Đầu tư công tăng có thể hỗ trợ tổng cầu, việc làm và tăng trưởng -> PD giảm; '
        'tuy nhiên tác động thường có độ trễ.'
    ],

    # =========================================================
    # 17. GOVERNMENT SPENDING
    # =========================================================
    [
        'GCE',
        'Government Spending',
        'Government Consumption Expenditures and Gross Investment',
        'Tổng chi tiêu tiêu dùng và đầu tư của khu vực chính phủ Mỹ.',
        '+/-',
        'Chi tiêu chính phủ tăng có thể hỗ trợ tăng trưởng và việc làm -> PD giảm; '
        'nhưng chi tiêu cũng thường tăng để phản ứng với suy thoái nên dấu thực nghiệm có thể không ổn định.'
    ],

    [
        'FGEXPND',
        'Government Spending',
        'Federal Government Current Expenditures',
        'Tổng chi tiêu hiện hành của chính phủ liên bang Mỹ.',
        '+/-',
        'Chi tiêu liên bang tăng có thể hỗ trợ thu nhập và nền kinh tế -> PD giảm; '
        'nhưng có thể tăng mạnh trong thời kỳ suy thoái nên tồn tại vấn đề phản ứng chính sách.'
    ],

    # =========================================================
    # 18. GOVERNMENT SAVING
    # =========================================================
    [
        'GGSAVE',
        'Government Saving',
        'Gross Government Saving',
        'Tiết kiệm gộp của khu vực chính phủ Mỹ.',
        '-',
        'Government saving tăng thường phản ánh vị thế tài khóa tốt hơn -> PD có xu hướng giảm.'
    ],

    # =========================================================
    # 19. GOVERNMENT DEBT
    # =========================================================
    [
        'GFDEGDQ188S',
        'Government Debt',
        'Federal Debt: Total Public Debt as Percent of Gross Domestic Product',
        'Tổng nợ công liên bang Mỹ tính theo tỷ lệ phần trăm GDP.',
        '+',
        'Nợ công/GDP tăng có thể phản ánh áp lực tài khóa hoặc điều kiện kinh tế yếu hơn -> PD có xu hướng tăng.'
    ],

    # =========================================================
    # 20. GOVERNMENT REVENUE
    # =========================================================
    [
        'FGRECPT',
        'Government Revenue',
        'Federal Government Current Receipts',
        'Tổng thu hiện hành của chính phủ liên bang Mỹ, bao gồm thuế và các nguồn thu hiện hành khác.',
        '+/-',
        'Thu ngân sách tăng có thể phản ánh kinh tế và thu nhập cải thiện -> PD giảm; '
        'nhưng thuế tăng cũng có thể làm giảm thu nhập khả dụng nên dấu không hoàn toàn xác định.'
    ],

    # =========================================================
    # 21. TREASURY INTEREST RATES
    # =========================================================
    [
        'TB3MS',
        'Interest Rates / Treasury',
        '3-Month Treasury Bill Secondary Market Rate',
        'Lợi suất tín phiếu Kho bạc Mỹ kỳ hạn 3 tháng, đại diện cho lãi suất phi rủi ro ngắn hạn.',
        '+',
        'Lợi suất ngắn hạn tăng thường gắn với điều kiện tiền tệ thắt chặt -> chi phí vốn tăng -> PD tăng.'
    ],

    [
        'DGS2',
        'Interest Rates / Treasury',
        'Market Yield on U.S. Treasury Securities at 2-Year Constant Maturity',
        'Lợi suất trái phiếu Kho bạc Mỹ kỳ hạn 2 năm theo phương pháp constant maturity.',
        '+/-',
        'Lợi suất 2 năm tăng thường phản ánh kỳ vọng lãi suất chính sách cao hơn và điều kiện tài chính chặt hơn -> PD tăng; '
        'tuy nhiên cũng có thể phản ánh kỳ vọng tăng trưởng hoặc lạm phát.'
    ],

    # =========================================================
    # 22. TAX / INCOME
    # =========================================================
    [
        'TLINCTX',
        'Tax / Income',
        'Individual Income Tax Filing: Total Income Tax',
        'Dữ liệu liên quan đến hồ sơ khai thuế thu nhập cá nhân; series có tần suất năm.',
        '+/-',
        'Không có quan hệ kinh tế trực tiếp và ổn định với consumer loan PD.'
    ]
]


# =========================================================
# CREATE DATAFRAME
# =========================================================

df_macro_info = pd.DataFrame(
    macro_info,
    columns=[
        'series',
        'group',
        'name',
        'definition',
        'expected_sign',
        'expected_relationship'
    ]
)


# =========================================================
# SORT BY GROUP
# =========================================================

df_macro_info = (
    df_macro_info
    .sort_values(
        by=['group', 'series']
    )
    .reset_index(drop=True)
)

In [68]:
# =========================================================
# CREATE LAG1, LAG2 INFORMATION
# =========================================================

df_lag1 = df_macro_info.copy()
df_lag1['series'] = df_lag1['series'] + '_lag1'
df_lag1['name'] = df_lag1['name'] + ' - Lag 1'
df_lag1['definition'] = df_lag1['definition'] + ' Giá trị trễ 1 quý.'
df_lag1['expected_relationship'] = (
    df_lag1['expected_relationship']
    + ' Sử dụng giá trị của quý trước (lag 1).'
)

df_lag2 = df_macro_info.copy()
df_lag2['series'] = df_lag2['series'] + '_lag2'
df_lag2['name'] = df_lag2['name'] + ' - Lag 2'
df_lag2['definition'] = df_lag2['definition'] + ' Giá trị trễ 2 quý.'
df_lag2['expected_relationship'] = (
    df_lag2['expected_relationship']
    + ' Sử dụng giá trị của 2 quý trước (lag 2).'
)

df_macro_info = pd.concat(
    [
        df_macro_info,
        df_lag1,
        df_lag2
    ],
    ignore_index=True
)
df_macro_info

,series,group,name,definition,expected_sign,expected_relationship
0,TERMCBCCALLNS,Consumer Credit Cost,Commercial Bank Interest Rate on Credit Card P...,Lãi suất áp dụng cho các tài khoản thẻ tín dụn...,+,Lãi suất tín dụng tiêu dùng tăng -> chi phí ph...
1,CORCACBS,Credit Risk,"Charge-Off Rate on Consumer Loans, All Commerc...",Tỷ lệ các khoản vay tiêu dùng bị charge-off tạ...,+,Charge-off rate tăng -> môi trường rủi ro tín ...
2,DRCLACBS,Credit Risk,"Delinquency Rate on Consumer Loans, All Commer...",Tỷ lệ các khoản vay tiêu dùng bị quá hạn tại c...,+,Delinquency của thị trường tăng -> chất lượng ...
3,POPTHM,Demographics,Population,Ước tính dân số Mỹ.,+/-,Quy mô dân số không có quan hệ trực tiếp rõ rà...
4,TTLHHM156N,Demographics,Household Estimates,Ước tính tổng số hộ gia đình tại Mỹ.,+/-,Số hộ gia đình không có dấu tác động trực tiếp...
...,...,...,...,...,...,...
94,TLINCTX_lag2,Tax / Income,Individual Income Tax Filing: Total Income Tax...,Dữ liệu liên quan đến hồ sơ khai thuế thu nhập...,+/-,Không có quan hệ kinh tế trực tiếp và ổn định ...
95,IC131_lag2,Trade / Transportation Prices,Inbound Price Index: Air Freight - Lag 2,Chỉ số giá dịch vụ vận chuyển hàng hóa bằng đư...,+/-,Quan hệ với consumer PD mang tính gián tiếp và...
96,IS231_lag2,Trade / Transportation Prices,Outbound Price Index: Air Freight - Lag 2,Chỉ số giá dịch vụ vận chuyển hàng hóa bằng đư...,+/-,Quan hệ với consumer PD mang tính gián tiếp và...
97,IQ_lag2,Trade Prices,Export Price Index (End Use): All Commodities ...,Chỉ số giá hàng hóa xuất khẩu của Mỹ. Giá trị ...,+/-,Tác động lên PD không rõ ràng vì phụ thuộc vào...


In [49]:
X = df_FED_pct[df_FED_pct.index <= pd.to_datetime('2019-12-31')].copy()
y = df_PD['ln_PD']
X.index = y.index
X.shape, y.shape

((40, 96), (40,))

In [50]:
df_full = pd.concat([df_FED_pct.reset_index(drop=True), df_PD[['ln_PD', 'PD']].reset_index(drop=True)], axis=1)

In [52]:
#Create X,y 
X_train = X[:-len_OOT]
y_train = y[:-len_OOT]
X_test = X[-len_OOT:]
y_test = y[-len_OOT:]
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((36, 96), (36,), (4, 96), (4,))

In [53]:
df_train = X.copy()
df_train[target] = y
corr = df_train.corr()
corr

,UNRATE,FEDFUNDS,CDSP,TDSP,DRCLACBS,CORCACBS,PSAVERT,TERMCBCCALLNS,FGEXPND,TB3MS,...,A782RC1Q027SBEA_lag2,GCE_lag1,GCE_lag2,GGSAVE_lag1,GGSAVE_lag2,GFDEGDQ188S_lag1,GFDEGDQ188S_lag2,FGRECPT_lag1,FGRECPT_lag2,ln_PD
UNRATE,1.0000,-0.7193,0.1225,0.8034,0.8112,0.7080,0.0849,-0.3126,-0.8497,-0.7078,...,-0.7184,-0.5986,-0.5342,-0.2530,-0.0022,0.8020,0.7916,0.3407,0.1219,0.4261
FEDFUNDS,-0.7193,1.0000,0.2127,-0.3687,-0.2827,-0.2095,0.2724,0.8231,0.9384,0.9959,...,0.8007,0.8421,0.8226,0.4273,0.2975,-0.3887,-0.3784,-0.3580,-0.2923,-0.4861
CDSP,0.1225,0.2127,1.0000,0.6729,0.5941,0.7185,-0.2079,0.5827,0.1650,0.2211,...,0.3324,0.5071,0.5377,0.5743,0.7260,0.4977,0.5447,-0.3543,-0.5322,0.3401
TDSP,0.8034,-0.3687,0.6729,1.0000,0.9676,0.9676,-0.0343,0.1703,-0.4961,-0.3565,...,-0.3065,-0.0907,-0.0262,0.1582,0.4290,0.8859,0.9058,0.0406,-0.2172,0.4507
DRCLACBS,0.8112,-0.2827,0.5941,0.9676,1.0000,0.9705,0.1265,0.2432,-0.4569,-0.2695,...,-0.3221,-0.0738,-0.0285,0.1708,0.4517,0.8983,0.9250,0.0179,-0.2599,0.3970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GFDEGDQ188S_lag1,0.8020,-0.3887,0.4977,0.8859,0.8983,0.8658,0.0878,0.1044,-0.5099,-0.3817,...,-0.3501,-0.1730,-0.1035,0.2294,0.3786,1.0000,0.9239,-0.0193,-0.2497,0.4360
GFDEGDQ188S_lag2,0.7916,-0.3784,0.5447,0.9058,0.9250,0.8988,0.0586,0.1134,-0.5153,-0.3738,...,-0.3774,-0.1476,-0.1200,0.3016,0.5198,0.9239,1.0000,-0.0537,-0.3235,0.5092
FGRECPT_lag1,0.3407,-0.3580,-0.3543,0.0406,0.0179,-0.0278,-0.1294,-0.3410,-0.3986,-0.3540,...,-0.5448,-0.3970,-0.3507,-0.7921,-0.5651,-0.0193,-0.0537,1.0000,0.7320,0.0158
FGRECPT_lag2,0.1219,-0.2923,-0.5322,-0.2172,-0.2599,-0.3341,-0.1370,-0.4195,-0.2497,-0.2919,...,-0.4224,-0.4403,-0.3421,-0.6723,-0.8461,-0.2497,-0.3235,0.7320,1.0000,-0.2286


In [54]:
write_file(folder_name = target[3:]+'_'+current_time+'_result/',file_name='Result_'+target[3:]+'_'+ str(num_var) +'_var'+'_'+current_time+'.xlsx',sheet_name='macro_info',data=df_macro_info)
write_file(folder_name = target[3:]+'_'+current_time+'_result/',file_name='Result_'+target[3:]+'_'+ str(num_var) +'_var'+'_'+current_time+'.xlsx',sheet_name='Full_Data',data=df_full)

In [77]:
df_macro_info = df_macro_info.set_index('series')

In [75]:
list(k)[0]

'UNRATE'

In [79]:
BP_col = []
SW_col = []
DW_col = []
ADF_col = []
Y_corr_col = []
R2_Adjust_col = []
R2_PD_train_col = []
MAPE_train_col = []
VIF_max_col = []
t_test_col = []
conservative_col = []
if len_OOT != 0:
    MAPE_test_col = []
    Y_OOT_corr_col = []
    conservative_OOT_col = []
    R2_Adjust_test_col = []
    z_test_col = []
else: 
    pass
var = []
corr_check = []
corr_check1 = []
intercept_col = []
coef_col = []
corr_check1 = []
for i in tqdm(list(combinations(X.columns, num_var))):
    #check corr between vars, check group var
    corr_var = []
    check_group = []
    for k in combinations(i, 2):
         corr_check1.append(corr[list(k)[0]][list(k)[1]])# meger corr 2 var each other
         corr_var.append(corr[list(k)[0]][list(k)[1]])
         check_group.append(df_macro_info['group'][list(k)[0]] != df_macro_info['group'][list(k)[1]])
    if all(abs(i)<=0.6 for i in corr_var) and all(i1==True for i1 in check_group):
      #create X,y and run model
      X_train_ = X_train[list(i)]
      X_train_ = sm.add_constant(X_train_)
      model = sm.OLS(y_train, X_train_).fit()
      df_coefficient = pd.read_html(model.summary().tables[1].as_html())[0]
      coefficients = float_list(list(df_coefficient.iloc[1:, 1]))
      coef = df_coefficient.iloc[2:2+num_var,1]
      coef = coef.reset_index(drop=True)
      coef_col.append(coef)
      intercept = df_coefficient.iloc[1,1]
      intercept_col.append(intercept)
      VIF_max = VIF_test(X_train_) 
      VIF_max_col.append(VIF_max)
      t_test = df_coefficient.iloc[2:2+num_var, 4]
      t_test_col.append(list(t_test))
      # Predict
      y_pred_train = model.predict(X_train_)
      #check conservative
      conservative = np.sum(y_pred_train >= y_train) * 1 / len(y_train)
      conservative_col.append(conservative)
      # check statistic
      residual = y_train.values.copy() - model.fittedvalues.copy() #model.residual
      _, BP, _, _ = het_breuschpagan(resid = residual, exog_het = model.model.exog)
      BP_col.append(BP)
      _, SW = shapiro(residual)
      SW_col.append(SW)
      cols = [c for c in X_train_.columns if c != 'const']
      rhs = ' + '.join(f'Q("{s.replace("\"","\\\"")}")' for s in cols)
      _, DW = dwtest(target + ' ~ ' + rhs, data = pd.concat([X_train_[cols], y_train], axis = 1))
      DW_col.append(DW)
      residual_ADF = adfuller(residual, autolag = 'AIC', regression = 'ct')[1]
      ADF_col.append(residual_ADF)
      Y_corr = y_train.corr(y_pred_train)
      Y_corr_col.append(Y_corr)
      # check R2 Ln_PD
      df_metric = pd.read_html(model.summary().tables[0].as_html())[0]
      SEs = float_list(list(df_coefficient.iloc[2:, 2]))
      R2_train = df_metric.iloc[0, 3]
      R2_Adjust = df_metric.iloc[1, 3]
      R2_Adjust_col.append(R2_Adjust)
      # Check R2 PD
      R2_PD_train = r2_score(y_pred=PD(y_pred_train), y_true=y_train.apply(PD))
      R2_PD_train_col.append(R2_PD_train)
      # Check MAPE PD
      MAPE_train = mean_absolute_percentage_error(y_pred=PD(y_pred_train), y_true=y_train.apply(PD))
      MAPE_train_col.append(MAPE_train)
      # Check MAE PD
      MAE_train = mean_absolute_error(y_pred=PD(y_pred_train), y_true=y_train.apply(PD))

      # Nếu len_OOT != 0, thêm các biến _test
      if len_OOT != 0:
         X_test_ = X_test[list(i)]
         X_test_ = sm.add_constant(X_test_)
         model_test = sm.OLS(y_test, X_test_).fit()
         R2_Adjust_test = float(pd.read_html(model_test.summary().tables[0].as_html())[0].iloc[1, 3])
         R2_Adjust_test_col.append(R2_Adjust_test)
         y_pred_test = model_test.predict(X_test_)
         MAPE_test = mean_absolute_percentage_error(y_pred=y_pred_test, y_true=y_test)
         MAPE_test_col.append(MAPE_test)
         conservative_OOT = np.sum(y_pred_test >= y_test) * 1 / len(y_test)
         conservative_OOT_col.append(conservative_OOT)
         Y_OOT_corr = y_test.corr(y_pred_test)
         Y_OOT_corr_col.append(Y_OOT_corr)
         df_test_coefficient = pd.read_html(model_test.summary().tables[1].as_html())[0]
         OOT_coefficients = float_list(list(df_test_coefficient.iloc[1:, 1]))
         OOT_SEs = float_list(list(df_test_coefficient.iloc[2:, 2]))
         z_test = []
         for e in range(len(SEs)):
            beta, OOT_beta = coefficients[e + 1], OOT_coefficients[e + 1]
            se_beta, OOT_se_beta = SEs[e], OOT_SEs[e]
            z_test.append((beta - OOT_beta) / np.sqrt(se_beta ** 2 + OOT_se_beta ** 2))
         z_test_col.append(z_test)
      # create corr and sign
      for j in range(len(i)):
         var.append(i[j])
         corr_check.append(corr[i[j]][target])# meger corr var and target
      for k in combinations(i, 2):
            corr_check.append(corr[list(k)[0]][list(k)[1]])# meger corr 2 var each other
      for j in range(len(i)): # meger sign corr var and target
         if (df_macro_info['expected_sign'][i[j]] == '+/-') or (df_macro_info['expected_sign'][i[j]] == '+' and float(coef[j])>0) or (df_macro_info['expected_sign'][i[j]] == '-' and float(coef[j])<0):
            corr_check.append('Yes')
         else: corr_check.append('No')

100%|██████████| 4560/4560 [08:57<00:00,  8.48it/s]


In [80]:
df_evaluation = pd.DataFrame()
# kiem dinh thong ke
t_test_col = np.array(t_test_col).reshape(-1,num_var)
df_t_test =  pd.DataFrame(t_test_col.astype(float), columns=['t_test_var_' + str(i+1) for i in range(num_var)])
df_evaluation['Breusch-Pagan_P_value'] = BP_col
df_evaluation['Shapiro-Wilk_P_value'] = SW_col
df_evaluation['Durbin-Watson_P_value'] = DW_col
df_evaluation['ADF_P_value'] = ADF_col
df_evaluation['Y_corr'] = Y_corr_col
df_evaluation['VIF_max'] = VIF_max_col

# Metric
df_evaluation['Conservative']=conservative_col
df_evaluation['Adjust_R2_train'] = R2_Adjust_col
df_evaluation['R2_PD_train'] = R2_PD_train_col
df_evaluation['MAPE_PD_train'] = MAPE_train_col


# Nếu len_OOT != 0, thêm các cột _OOT (R2_test, R2_PD_test, MAPE_test, MAE_test)
if len_OOT != 0:
    df_evaluation['Adjust_R2_OOT'] = R2_Adjust_test_col
    df_evaluation['MAPE_PD_OOT'] = MAPE_test_col
    df_evaluation['conservative_OOT'] = conservative_OOT_col
    df_evaluation['Y_OOT_corr'] = Y_OOT_corr_col
    Z_test_ = np.array(z_test_col).reshape(-1,num_var)
    df_Z_test =  pd.DataFrame(Z_test_, columns=['Z_score_var_' + str(i+1) for i in range(num_var)])
    df_evaluation = pd.merge(df_evaluation, df_Z_test, left_index=True, right_index=True)

# Merge kết quả với df_t_test
df_evaluation = pd.merge(df_t_test, df_evaluation, left_index=True, right_index=True)


In [81]:
df_evaluation

,t_test_var_1,t_test_var_2,Breusch-Pagan_P_value,Shapiro-Wilk_P_value,Durbin-Watson_P_value,ADF_P_value,Y_corr,VIF_max,Conservative,Adjust_R2_train,R2_PD_train,MAPE_PD_train,Adjust_R2_OOT,MAPE_PD_OOT,conservative_OOT,Y_OOT_corr,Z_score_var_1,Z_score_var_2
0,0.1140,0.0250,0.2919,0.0013,0.0013,0.9594,0.4771,1.0299,0.4444,0.1810,0.2924,0.1036,0.6810,0.0106,0.5000,0.9454,2.9839,2.2249
1,0.0350,0.1890,0.0769,0.0183,0.0070,0.5549,0.3817,1.0460,0.4444,0.0940,0.1759,0.1081,0.9730,0.0029,0.2500,0.9955,10.6600,-3.5369
2,0.0680,0.7900,0.0106,0.0536,0.0004,0.9562,0.3178,1.0030,0.4444,0.0470,0.1197,0.1113,0.7790,0.0084,0.5000,0.9624,3.2852,-1.9705
3,0.0990,0.6210,0.0644,0.0404,0.0006,0.9897,0.3252,1.0846,0.4722,0.0520,0.1265,0.1102,0.6550,0.0111,0.5000,0.9408,2.8797,0.5608
4,0.0280,0.1130,0.0988,0.0621,0.0012,0.0000,0.4074,1.0560,0.5000,0.1150,0.1907,0.1115,0.7840,0.0083,0.5000,0.9634,2.8589,0.7373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3558,0.0160,0.2350,0.7470,0.0817,0.0069,0.0057,0.4065,1.4663,0.5278,0.1150,0.2120,0.1183,0.9800,0.0026,0.5000,0.9967,5.6569,-9.1615
3559,0.0120,0.8660,0.0641,0.0012,0.0041,0.9592,0.4201,1.0010,0.3889,0.1270,0.2333,0.1107,0.3460,0.0160,0.5000,0.8842,1.4539,-0.5454
3560,0.0280,0.2600,0.0292,0.0045,0.0041,0.9524,0.4554,1.0764,0.4444,0.1590,0.2752,0.1120,0.2530,0.0147,0.2500,0.8667,2.3343,-0.4897
3561,0.0030,0.9820,0.2343,0.0010,0.0054,0.9805,0.4890,1.0070,0.3611,0.1930,0.3037,0.1063,0.9900,0.0018,0.5000,0.9984,8.9242,-12.2094


In [82]:
var_ = np.array(var).reshape(-1, num_var)
df_var = pd.DataFrame(var_, columns=['var_' + str(i+1) for i in range(num_var)])
corr_check_ = np.array(corr_check).reshape(-1, num_var + math.comb(num_var,2) + num_var)
col_corr = ['corr_var_' + str(i+1) + '_dependent' for i in range(num_var)]+['corr_'+ str(i) for i in combinations(range(1,num_var+1), 2)]+['Sign_var_' + str(i+1) + '_dependent' for i in range(num_var)]
df_corr = pd.DataFrame(corr_check_, columns=col_corr)
coef_ = np.array(coef_col).reshape(-1,num_var)
df_coef =  pd.DataFrame(coef_.astype(float), columns=['coef_var_' + str(i+1) for i in range(num_var)])
df_coef['Intercept'] = [float(i) for i in intercept_col]

In [83]:
df_corr['corr_var_1_dependent'] = df_corr['corr_var_1_dependent'].astype(float)
df_corr['corr_var_2_dependent'] = df_corr['corr_var_2_dependent'].astype(float)
df_corr['corr_(1, 2)'] = df_corr['corr_(1, 2)'].astype(float)

In [84]:
#final result
df_result = pd.merge(df_var, df_coef, left_index=True, right_index=True)
df_result = pd.merge(df_result, df_corr, left_index=True, right_index=True)
df_result = pd.merge(df_result, df_evaluation, left_index=True, right_index=True)

In [90]:
# write combinations into excel
combination = [i for i in combinations(df_train.columns.drop(target), num_var)]
col = ['Var_'+str(i+1) for i in range(num_var)]
df_comb = pd.DataFrame(combination, columns=col)
df_comb['Correlation'] = corr_check1
write_file(folder_name = target[3:]+'_'+current_time+'_result/',file_name='Result_'+target[3:]+'_'+ str(num_var) +'_var'+'_'+current_time+'.xlsx',sheet_name='combinations',data=df_comb)
# df_comb.to_excel('output/'+target[:-3]+'_'+str(num_var)+'/combinations_'+str(num_var)+'.xlsx')

In [91]:
#write final result into excel
write_file(folder_name = target[3:]+'_'+current_time+'_result/',file_name='Result_'+target[3:]+'_'+ str(num_var) +'_var'+'_'+current_time+'.xlsx',sheet_name='Full_result',data=df_result)